In [1]:
import os, sys, gc, cv2, timm, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from xgboost.callback import EarlyStopping
import xgboost as xgb



/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
# --- CONFIG ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 
MODEL_NAME = 'eva02_large_patch14_448'
EVA_WEIGHT_PATH = '/kaggle/input/aqaqaq/eva02_large_weights.pth'
DATA_DIR = "/kaggle/input/csiro-biomass"

In [3]:
# 1. สร้างโครงโมเดลและยืด Weight (Resampling)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
if os.path.exists(EVA_WEIGHT_PATH):
    sd = torch.load(EVA_WEIGHT_PATH, map_location="cpu")
    sd = sd.get("model", sd.get("state_dict", sd))
    if 'pos_embed' in sd:
        pe = sd['pos_embed']
        orig_s, new_s = int((pe.shape[1]-1)**0.5), int((model.pos_embed.shape[1]-1)**0.5)
        if orig_s != new_s:
            print(f"🔄 Resampling Position Embeddings: {orig_s} -> {new_s}")
            tok, pos = pe[:, :1], pe[:, 1:]
            pos = F.interpolate(pos.reshape(-1, orig_s, orig_s, 1024).permute(0,3,1,2), size=(new_s, new_s), mode='bicubic')
            sd['pos_embed'] = torch.cat((tok, pos.permute(0,2,3,1).flatten(1, 2)), dim=1)
    model.load_state_dict(sd, strict=False)
    print("✅ EVA-02 Model Ready!")

model.to(DEVICE).eval()

# 2. AdamW Optimizer (เตรียมไว้เพื่อเสถียรภาพ)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.05)

# 3. Image Transform
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
    transforms.Normalize(mean=model.default_cfg['mean'], std=model.default_cfg['std'])
])

🔄 Resampling Position Embeddings: 16 -> 32
✅ EVA-02 Model Ready!


In [4]:
# 1. กำหนดค่าเป้าหมาย
TARGET_NAMES = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHT_MAP = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1, "GDM_g": 0.2, "Dry_Total_g": 0.5}

# 2. โหลดไฟล์และจัดโครงสร้างแบบกว้าง (Wide Format)
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
train_df["image_id"] = train_df["sample_id"].str.split("__").str[0]
test_df["image_id"]  = test_df["sample_id"].str.split("__").str[0]

train_wide = train_df.pivot(index="image_id", columns="target_name", values="target").reset_index()
path_map = train_df.set_index('image_id')['image_path'].to_dict()
train_wide['image_path'] = train_wide['image_id'].map(path_map).apply(lambda x: os.path.join(DATA_DIR, x))

test_images = test_df[['image_id', 'image_path']].drop_duplicates().reset_index(drop=True)
test_images['image_path'] = test_images['image_path'].apply(lambda x: os.path.join(DATA_DIR, x))

Y_train = train_wide[TARGET_NAMES].values
print(f"✅ Data Ready: train_wide {train_wide.shape}")

✅ Data Ready: train_wide (357, 7)


In [5]:
class ImageDataset(Dataset):
    def __init__(self, df, transform):
        self.paths, self.ids = df["image_path"].values, df["image_id"].values
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        return self.ids[idx], self.transform(Image.open(self.paths[idx]).convert("RGB"))



In [6]:
def extract_deep_feats_tta(df, name):
    loader = DataLoader(ImageDataset(df, transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    feats = []
    with torch.no_grad():
        for _, imgs in tqdm(loader, desc=name):
            imgs = imgs.to(DEVICE)
            # TTA: มอง 2 มุม (ปกติ + พลิกซ้ายขวา)
            out1 = model.forward_features(imgs)
            out2 = model.forward_features(torch.flip(imgs, dims=[3]))
            # ดึง CLS และ AVG
            def get_f(o): return torch.cat([o[:, 0], torch.mean(o[:, 1:], 1)], 1)
            feats.append(((get_f(out1) + get_f(out2)) / 2).cpu().numpy())
    return np.vstack(feats)



In [7]:
train_emb = extract_deep_feats_tta(train_wide, "สกัด Deep Feature (Train)")
test_emb  = extract_deep_feats_tta(test_images, "สกัด Deep Feature (Test)")


สกัด Deep Feature (Train):   0%|          | 0/45 [00:00<?, ?it/s]

สกัด Deep Feature (Test):   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
def get_color_feats(df):
    feats = []
    for p in tqdm(df['image_path'], desc="สกัดค่าสี"):
        img = cv2.imread(p)
        if img is None: feats.append(np.zeros(10)); continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        img_n = cv2.resize(rgb, (224,224)).astype(float)/255.0
        exg = 2*img_n[:,:,1] - img_n[:,:,0] - img_n[:,:,2]
        green = cv2.inRange(cv2.resize(hsv, (224,224)), np.array([25,40,40]), np.array([85,255,255]))
        # เก็บสถิติ 10 ค่า
        feats.append([np.mean(exg), np.std(exg), np.sum(green>0)/(224*224)] + [0]*7)
    return np.array(feats)

X_train_color = get_color_feats(train_wide)
X_test_color  = get_color_feats(test_images)

สกัดค่าสี:   0%|          | 0/357 [00:00<?, ?it/s]

สกัดค่าสี:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
# นำ AI Features (2048) + Color Features (10) = 2058 มิติ
X_train = np.hstack([train_emb, X_train_color])
X_test  = np.hstack([test_emb,  X_test_color])

print("-" * 30)
print(f"🔥 FINAL INPUT SHAPE: {X_test.shape} (Defined Successfully!)")
print("-" * 30)

------------------------------
🔥 FINAL INPUT SHAPE: (1, 2058) (Defined Successfully!)
------------------------------


In [10]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)  # เก็บ 95% variance
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "max_depth": 2,
    "eta": 0.03,
    "subsample": 0.7,
    "colsample_bytree": 0.4,
    "min_child_weight": 20,
    "gamma": 2.0,
    "alpha": 10.0,
    "lambda": 20.0,
    "tree_method": "hist",
    "seed": 42,
}

groups = train_wide["image_id"].values
gkf = GroupKFold(n_splits=5)

oof_preds = np.zeros_like(Y_train)
test_preds = np.zeros((X_test.shape[0], len(TARGET_NAMES)))
fold_rmses = []

for t_idx, target in enumerate(TARGET_NAMES):
    y = Y_train[:, t_idx]

    for fold, (tr_idx, va_idx) in enumerate(
        gkf.split(X_train, y, groups)
    ):
        # ===== Split =====
        X_tr_pca = X_train_pca[tr_idx]
        X_va_pca = X_train_pca[va_idx]

        X_tr = X_train[tr_idx]
        X_va = X_train[va_idx]

        y_tr, y_va = y[tr_idx], y[va_idx]

        # ===== XGB (PCA) =====
        dtrain = xgb.DMatrix(X_tr_pca, label=y_tr)
        dvalid = xgb.DMatrix(X_va_pca, label=y_va)
        dtest  = xgb.DMatrix(X_test_pca)

        model_xgb = xgb.train(
            params,
            dtrain,
            num_boost_round=600,
            evals=[(dvalid, "val")],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        val_pred_xgb = model_xgb.predict(dvalid)
        test_pred_xgb = model_xgb.predict(dtest)

        # ===== Ridge (Full feat) =====
        model_ridge = make_pipeline(
            StandardScaler(),
            Ridge(alpha=30)
        )
        model_ridge.fit(X_tr, y_tr)

        val_pred_ridge = model_ridge.predict(X_va)
        test_pred_ridge = model_ridge.predict(X_test)

        # ===== Blend =====
        val_pred = 0.7 * val_pred_xgb + 0.3 * val_pred_ridge
        test_pred = 0.7 * test_pred_xgb + 0.3 * test_pred_ridge

        oof_preds[va_idx, t_idx] = val_pred
        test_preds[:, t_idx] += test_pred / gkf.n_splits

        rmse = np.sqrt(np.mean((y_va - val_pred) ** 2))
        fold_rmses.append(rmse)

        print(f"[Target {target} | Fold {fold}] RMSE = {rmse:.4f}")



Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 291, in _feed
    queue_sem.release()
ValueError: semaphore or lock released too many times


[Target Dry_Green_g | Fold 0] RMSE = 14.6055
[Target Dry_Green_g | Fold 1] RMSE = 12.5402
[Target Dry_Green_g | Fold 2] RMSE = 17.0099
[Target Dry_Green_g | Fold 3] RMSE = 13.8617
[Target Dry_Green_g | Fold 4] RMSE = 13.6167
[Target Dry_Dead_g | Fold 0] RMSE = 9.4358
[Target Dry_Dead_g | Fold 1] RMSE = 8.8187
[Target Dry_Dead_g | Fold 2] RMSE = 6.7691
[Target Dry_Dead_g | Fold 3] RMSE = 7.4865
[Target Dry_Dead_g | Fold 4] RMSE = 12.0207
[Target Dry_Clover_g | Fold 0] RMSE = 6.6274
[Target Dry_Clover_g | Fold 1] RMSE = 4.8699
[Target Dry_Clover_g | Fold 2] RMSE = 6.6252
[Target Dry_Clover_g | Fold 3] RMSE = 9.9571
[Target Dry_Clover_g | Fold 4] RMSE = 6.7025
[Target GDM_g | Fold 0] RMSE = 13.1501
[Target GDM_g | Fold 1] RMSE = 12.2340
[Target GDM_g | Fold 2] RMSE = 17.6691
[Target GDM_g | Fold 3] RMSE = 14.6876
[Target GDM_g | Fold 4] RMSE = 13.4478
[Target Dry_Total_g | Fold 0] RMSE = 15.7989
[Target Dry_Total_g | Fold 1] RMSE = 14.1293
[Target Dry_Total_g | Fold 2] RMSE = 19.2144
[Tar

In [11]:
for i, rmse in enumerate(fold_rmses):
    print(f"Fold {i+1}: RMSE = {rmse:.4f}")

print("Mean RMSE:", np.mean(fold_rmses))
print("Std RMSE :", np.std(fold_rmses))


Fold 1: RMSE = 14.6055
Fold 2: RMSE = 12.5402
Fold 3: RMSE = 17.0099
Fold 4: RMSE = 13.8617
Fold 5: RMSE = 13.6167
Fold 6: RMSE = 9.4358
Fold 7: RMSE = 8.8187
Fold 8: RMSE = 6.7691
Fold 9: RMSE = 7.4865
Fold 10: RMSE = 12.0207
Fold 11: RMSE = 6.6274
Fold 12: RMSE = 4.8699
Fold 13: RMSE = 6.6252
Fold 14: RMSE = 9.9571
Fold 15: RMSE = 6.7025
Fold 16: RMSE = 13.1501
Fold 17: RMSE = 12.2340
Fold 18: RMSE = 17.6691
Fold 19: RMSE = 14.6876
Fold 20: RMSE = 13.4478
Fold 21: RMSE = 15.7989
Fold 22: RMSE = 14.1293
Fold 23: RMSE = 19.2144
Fold 24: RMSE = 14.5542
Fold 25: RMSE = 16.8708
Mean RMSE: 12.1081269547195
Std RMSE : 3.93610784318519


In [12]:
train_pred = model_xgb.predict(dtrain)
train_rmse = np.sqrt(np.mean((train_pred - y_tr) ** 2))
print("Train RMSE:", train_rmse)


Train RMSE: 9.754888336897281


In [13]:
sub_rows = []
for i, row in test_images.iterrows():
    for t_idx, target in enumerate(TARGET_NAMES):
        sub_rows.append([
            f"{row['image_id']}__{target}",
            max(0, test_preds[i, t_idx])
        ])

pd.DataFrame(sub_rows, columns=['sample_id', 'target']) \
  .to_csv("submission.csv", index=False)
